# Preprocessing Data
I retrieved a dataset containing annotations in a csv format, and so I basically convert it into a yolo-friendly format using yaml and the correct structure for directories and subdirectories to get the model up and running. 

In [1]:
import pandas as pd
import os
import random
import shutil

In [2]:
train_data = pd.read_csv("./train.csv")
test_data = pd.read_csv("./test.csv")
categories = pd.read_csv("./categories.csv")
image_ids = pd.read_csv("./image_ids.csv")

In [3]:
categories.head()

,id,name
0,1,signature
1,2,initials
2,3,redaction
3,4,date


In [4]:
image_ids.head()

,height,width,id,file_name
0,3300.0,2560.0,1,nist_r0392_01.png
1,3300.0,2560.0,2,nist_r0647_01.png
2,4454.0,3480.0,3,gsa_LAL50113-Lease-_Z-03.png
3,3300.0,2560.0,4,nist_r0169_01.png
4,4411.0,3422.0,5,gsa_(R)LDC02012-Lease-SF2-04.png


In [5]:
train_data.head()

,area,bbox,category_id,id,image_id
0,0.009,"[0.14669516046674358, 0.8172596266496202, 0.33...",1,4,2
1,0.010,"[0.1830540707236842, 0.8483832465277777, 0.293...",1,5,2
2,0.008,"[0.21358420632102273, 0.8757179542824074, 0.23...",1,6,2
3,0.006,"[0.5366111738148984, 0.899244842346794, 0.0856...",2,7,3
4,0.005,"[0.6299114700902935, 0.90049396494709, 0.06553...",2,8,3


From this, we can gather that essentially each image corresponds to an `image_id` in the `image_ids.csv`, and each `image_id` then corresponds to set of coordinates in the `train.csv` and the `test.csv` data, under the `bbox` column.

Thus, we write a script that indexes these and then formats it into the correct substructure. 

# Re-structuring
1. Find the image id, and get its correspondiong dataset
2. Create the correct structure of directories
3. Add a label for each of the images

In [6]:
# finds the image id
def get_image_id(filename):
    return image_ids[image_ids["file_name"] == filename].iloc[0]['id'].tolist()

# retrieves the bbox for a given image; returns None if no annotations are provided for a given document
def get_bbox(image_id):
    train_bbox = train_data[train_data["image_id"] == image_id]
    test_bbox = test_data[test_data["image_id"] == image_id]

    try:
        if train_bbox is not None:
            return eval(train_bbox.iloc[0]['bbox'])
        else:
            return eval(test_bbox.iloc[0]['bbox'])
    except Exception as e:
        return None

In [7]:
# split the dataset
images = [f for f in os.listdir("./images") if f.endswith(".png")]
random.shuffle(images)

dir_len = len(images)
train_split = int(0.7 * dir_len)
test_split = int(0.9 * dir_len)

train_imgs = images[:train_split] # 70%
test_imgs = images[train_split:test_split] # 20%
valid_imgs = images[test_split:] # 10%

splits = {
    "train": train_imgs,
    "test":  test_imgs,
    "valid": valid_imgs
}

ROOT_DIR = "signature-detection-2"
for split, img_list in splits.items():
    for img in img_list:
        image_path = f'./{ROOT_DIR}/{split}/images/{img}'
        os.makedirs(os.path.dirname(image_path), exist_ok=True)
        shutil.copy(f"./images/{img}", image_path)

In [8]:
def generate_labels(split, img_list):
    for img in img_list:
        bbox = get_bbox(get_image_id(img))
        if bbox is None: # aka part of the testing dataset
            bbox_formatted = ""
        else:
            bbox_formatted = f'0 {" ".join(map(str, bbox))}'
            
        label_filename = f'./{ROOT_DIR}/{split}/labels/{img[:-4]}.txt'
        os.makedirs(os.path.dirname(label_filename), exist_ok=True)

        with open(label_filename, "w") as f:
            f.write(bbox_formatted)

In [9]:
# Generate labels for each split
generate_labels("train", train_imgs)
generate_labels("test", test_imgs)
generate_labels("valid", valid_imgs)

Lastly, we create the `data.yaml` file necessary for telling `YOLO` what the structure of our dataset looks like. 

In [10]:
# create yaml file
os.path.join(f'./{ROOT_DIR}', 'data.yaml')
with open(f'./{ROOT_DIR}/data.yaml', "w") as f:
    f.write(
"""
names:
- signature
nc: 1
test: ../test/images
train: ../train/images
val: ../valid/images
"""
    )

## Checking structure
When it's all said and done, the structure should look something resemble that of a `YOLO` format, which does seem to be the case. And so, we are done reformatting our code. 

In [30]:
def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        base_name = os.path.basename(root)
        print(f'{indent}{base_name}/')
        subindent = ' ' * 4 * (level + 1)

        if base_name == 'images' or base_name == 'labels':
            print(f'{subindent}   ...')
list_files(f"./{ROOT_DIR}")

signature-detection-2/
    valid/
        images/
               ...
        labels/
               ...
    test/
        images/
               ...
        labels/
               ...
    train/
        images/
               ...
        labels/
               ...
